# **1. Setup**



In [1]:
!pip install transformers datasets scikit-learn torchinfo evaluate -q

import torch
import evaluate
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset, Dataset
from torchinfo import summary
from torch.optim import AdamW
from transformers import BertTokenizer, BertForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from collections import defaultdict, Counter
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00


# **2. Data Acquisition**



In [2]:
# Load MedHallu datasets
train_dataset = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_artificial")['train']
labeled_dataset = load_dataset("UTAustin-AIHealth/MedHallu", "pqa_labeled")['train']

# Subsample labeled dataset for val/test split. 50% validation 50% test. 1000 rows -> 500 val, 500 test
labeled_list = [dict(x) for x in labeled_dataset]
val_orig, test_orig = train_test_split(labeled_list, test_size=0.5, random_state=42)

# Convert artificial dataset to list for preprocessing
train_list = [dict(x) for x in train_dataset]

print(f"Train size: {len(train_list)}, Val size: {len(val_orig)}, Test size: {len(test_orig)}")

# Load Med-HALT datasets
medhalt_fct = load_dataset("openlifescienceai/Med-HALT", "reasoning_FCT")['train']
print("medhalt_fct columns: " , medhalt_fct.column_names)
print(medhalt_fct[0])

medhalt_nota = load_dataset("openlifescienceai/Med-HALT", "reasoning_nota")['train']
print("medhalt_nota columns: " , medhalt_nota.column_names)
print(medhalt_nota[0])

medhalt_fake = load_dataset("openlifescienceai/Med-HALT", "reasoning_fake")['train']
print("medhalt_fake columns: " , medhalt_fake.column_names)
print(medhalt_fake[0])



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pqa_artificial/train-00000-of-00001.parq(…):   0%|          | 0.00/10.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9000 [00:00<?, ? examples/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/18866 [00:00<?, ? examples/s]

medhalt_nota columns:  ['id', 'dataset', 'question', 'options', 'correct_answer', 'correct_index', 'split_type', 'subject_name', 'topic_name', 'year', 'exam_name']
{'id': '07d9bcf9-feb6-4d4a-858e-5570c5d9ef8b', 'dataset': 'medmcqa', 'question': 'Zone 3 is known as:', 'options': "{'0': 'Bacterial zone.', '1': 'Neutrophil rich zone.', '2': 'None of the above', '3': 'Spirochetial infiltration zone.'}", 'correct_answer': 'None of the above', 'correct_index': 2, 'split_type': 'test', 'subject_name': 'Dental', 'topic_name': None, 'year': None, 'exam_name': None}


reasoning_fake.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1858 [00:00<?, ? examples/s]

medhalt_fake columns:  ['id', 'subject_name', 'topic_name', 'split_type', 'dataset', 'len', 'year', 'exam_name', 'question', 'options']
{'id': '9d587cba-4d1e-4256-9315-10054380901e', 'subject_name': 'chemistry', 'topic_name': None, 'split_type': 'train', 'dataset': 'headqa_en', 'len': 60, 'year': 2013.0, 'exam_name': 'Cuaderno_2013_1_Q', 'question': 'In a bizarre and bewildering alternate universe, where the laws of nature are turned on their head, a perplexing and unknown process exists which is the antithesis of gluconeogenesis. In this surreal and illogical world, which of the following utterly incomprehensible statements is NOT true about this mysterious process?', 'options': "{'0': 'Through a series of arcane and inexplicable enzymatic reactions, vastly different from the arcane alchemy of glycolysis, this process is brought to fruition', '1': 'This process is the convoluted and utterly ridiculous synthesis of glucose from precursors that are already carbohydrates, because who nee

In [3]:
# Look at the first few examples of Medhallu
for i in range(4):
  print("Question: ", train_dataset['Question'][i])
  print("Knowledge: ", train_dataset['Knowledge'][i])
  print("Ground Truth: ", train_dataset['Ground Truth'][i])
  print("Hallucinated Answer: ", train_dataset['Hallucinated Answer'][i])
  print("Category of Hallucination: ", train_dataset['Category of Hallucination'][i])
  print()

# Look at the categories
# Extract all category values
categories = train_dataset["Category of Hallucination"]  # or dataset_val / dataset_test

# Get unique categories
unique_categories = sorted(set(categories))

print(unique_categories)
print(f"Total categories: {len(unique_categories)}")

Question:  Are group 2 innate lymphoid cells ( ILC2s ) increased in chronic rhinosinusitis with nasal polyps or eosinophilia?
Knowledge:  ['Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated in driving Th2 inflammation in CRS; however, their relationship with clinical disease characteristics has yet to be investigated.', 'The aim of this study was to identify ILC2s in sinus mucosa in patients with CRS and controls and compare ILC2s across characteristics of disease.', 'A cross-sectional study of patients with CRS undergoing endoscopic sinus surgery was conducted. Sinus mucosal biopsies were obtained during surgery and control tissue from patients undergoing pituitary tumour resection through transphenoidal approach. ILC2s were identified as CD45(+) Lin(-) CD127(+) CD4(-) CD8(-) CRTH2(CD294)(+) CD161(+) cells in single cell suspensions thro

# **3. Bert Tokenizer**

In [4]:
checkpoint = "bert-base-uncased"
bert_tokenizer = BertTokenizer.from_pretrained(checkpoint)

max_length = 256

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# **4. Preprocess Medhallu dataset**



In [5]:
def preprocess_medhallu(example):

    processed_examples = []

    knowledge_list = example["Knowledge"]
    question = example["Question"]
    gt_answer = example["Ground Truth"]
    hallucinated_answer = example["Hallucinated Answer"]
    category = example["Category of Hallucination"]

    if not gt_answer or not hallucinated_answer:
      return []

    # Convert knowledge list to string
    if isinstance(knowledge_list, list):
        knowledge = " ".join(knowledge_list)
    else:
        knowledge = knowledge_list  # fallback (just in case)

    # include question for richer context
    premise = f"Question: {question} Context: {knowledge}"

    # Positive example (entailed)
    processed_examples.append({
        "text_a": premise,
        "text_b": gt_answer,
        "label": 0,
        "category": category
    })

    # Negative example (hallucinated)
    processed_examples.append({
        "text_a": premise,
        "text_b": hallucinated_answer,
        "label": 1,
        "category": category
    })

    return processed_examples

def build_dataset(split):
    processed = []
    for s in split:
        processed.extend(preprocess_medhallu(s))
    return processed


processed_medhallu_train = build_dataset(train_list)
processed_medhallu_val   = build_dataset(val_orig)
processed_medhallu_test  = build_dataset(test_orig)

dataset_medhallu_train = Dataset.from_list(processed_medhallu_train)
dataset_medhallu_val = Dataset.from_list(processed_medhallu_val)
dataset_medhallu_test = Dataset.from_list(processed_medhallu_test)

# **4. Preprocess Medhalt dataset**

In [6]:
"""
False Confidence Test (FCT)
The False Confidence Test (FCT) involves presenting a multiple-choice medical question and a randomly suggested correct answer to the language model, tasking it with evaluating the validity of the proposed answer and providing detailed explanations for its correctness or incorrectness, in addition to explaining why the other options are wrong.
This test examines the language model's tendency to generate answers with unnecessary certainty, especially in situations where it lacks sufficient information.
"""

def preprocess_medhalt_fct(example):

    processed = []

    question = example["question"]
    correct = example["correct_answer"]
    student = example["student_answer"]

    correct_idx = example["correct_index"]
    student_idx = example["student_index"]

    if not question or not correct or not student:
        return []

    premise = f"Question: {question}"

    # Correct answer - NOT hallucinated
    processed.append({
        "text_a": premise,
        "text_b": correct,
        "label": 0,
        "category": "FCT"
    })

    # Student answer - depends on correctness
    label = 0 if student_idx == correct_idx else 1

    processed.append({
        "text_a": premise,
        "text_b": student,
        "label": label,
        "category": "FCT"
    })

    return processed


"""
None of the Above Test (Nota)
In the None of the Above (Nota) Test, the model is presented with a multiple-choice medical question where the correct answer is replaced by 'None of the above', requiring the model to identify this and justify its selection.
It tests the model's ability to distinguish irrelevant or incorrect information.
"""

import ast

def preprocess_medhalt_nota(example):

    processed = []

    question = example["question"]
    options = ast.literal_eval(example["options"])
    correct_idx = example["correct_index"]

    if not question:
        return []

    premise = f"Question: {question}"

    for idx, ans in options.items():

        label = 0 if int(idx) == correct_idx else 1

        processed.append({
            "text_a": premise,
            "text_b": ans,
            "label": label,
            "category": "NOTA"
        })

    return processed

"""
Fake Questions Test (FQT)
This test involves presenting the model with fake or nonsensical medical questions to examine whether it can correctly identify and handle such queries.
We employed a hybrid approach for generating fake questions, where a subset was crafted by human experts, while the remaining were generated using GPT-3.5.
"""

def preprocess_medhalt_fake(example):

    processed = []

    question = example["question"]

    if not question:
        return []

    premise = f"Question: {question}"

    processed.append({
        "text_a": premise,
        "text_b": "This question is invalid or nonsensical.",
        "label": 1,
        "category": "FAKE"
    })

    return processed

# Build Med-HALT dataset
def build_medhalt_dataset():

    processed = []

    for x in medhalt_fct:
        processed.extend(preprocess_medhalt_fct(dict(x)))

    for x in medhalt_nota:
        processed.extend(preprocess_medhalt_nota(dict(x)))

    for x in medhalt_fake:
        processed.extend(preprocess_medhalt_fake(dict(x)))

    return Dataset.from_list(processed)


dataset_medhalt_test = build_medhalt_dataset()
print("Med-Halt orginal size: " ,dataset_medhalt_test)
#dataset_medhalt_test = dataset_medhalt_test.select(range(20000))
#print("Med-Halt subsample size: " ,dataset_medhalt_test)


Med-Halt orginal size:  Dataset({
    features: ['text_a', 'text_b', 'label', 'category'],
    num_rows: 114886
})


# **5. Tokenization**

In [7]:
def tokenize(example):
    return bert_tokenizer(
        example["text_a"],
        example["text_b"],
        truncation=True,
        padding="max_length",
        max_length=max_length
    )

# Tokenize Medhallu
dataset_medhallu_train = dataset_medhallu_train.map(tokenize, batched=True)
dataset_medhallu_val = dataset_medhallu_val.map(tokenize, batched=True)
dataset_medhallu_test = dataset_medhallu_test.map(tokenize, batched=True)

# Tokenize Med-HALT
dataset_medhalt_test = dataset_medhalt_test.map(tokenize, batched=True)

Map:   0%|          | 0/18000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/114886 [00:00<?, ? examples/s]

# **6. Build final dataset**

In [8]:
# Build Medhallu dataset
dataset_medhallu_train = dataset_medhallu_train.remove_columns(["text_a", "text_b"])
dataset_medhallu_train.set_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])

dataset_medhallu_val = dataset_medhallu_val.remove_columns(["text_a", "text_b"])
val_medhallu_categories = dataset_medhallu_val["category"]
print("\nMedHallu Category Distribution (Validation):")
print(Counter(val_medhallu_categories))
dataset_medhallu_val.set_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])

dataset_medhallu_test = dataset_medhallu_test.remove_columns(["text_a", "text_b"])
test_medhallu_categories = dataset_medhallu_test["category"]
print("\nMedHallu Category Distribution (Test):")
print(Counter(test_medhallu_categories))
dataset_medhallu_test.set_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])

# Build Med-HALT dataset
dataset_medhalt_test = dataset_medhalt_test.remove_columns(["text_a", "text_b"])
test_medhalt_categories = dataset_medhalt_test["category"]
print("\nMed-HALT Category Distribution (Test):")
print(Counter(test_medhalt_categories))
dataset_medhalt_test.set_format(type="torch", columns=["input_ids", "attention_mask", "token_type_ids", "label"])

print("\ndataset_medhallu_train:")
print(dataset_medhallu_train)

print("\ndataset_medhallu_val:")
print(dataset_medhallu_val)

print("\ndataset_medhallu_test:")
print(dataset_medhallu_test)

print("\ndataset_medhalt_test:")
print(dataset_medhalt_test)


MedHallu Category Distribution (Validation):
Counter({'Misinterpretation of #Question#': 762, 'Incomplete Information': 202, 'Mechanism and Pathway Misattribution': 32, 'Methodological and Evidence Fabrication': 4})

MedHallu Category Distribution (Test):
Counter({'Misinterpretation of #Question#': 742, 'Incomplete Information': 222, 'Mechanism and Pathway Misattribution': 34, 'Methodological and Evidence Fabrication': 2})

Med-HALT Category Distribution (Test):
Counter({'NOTA': 75464, 'FCT': 37564, 'FAKE': 1858})

dataset_medhallu_train:
Dataset({
    features: ['label', 'category', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 18000
})

dataset_medhallu_val:
Dataset({
    features: ['label', 'category', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1000
})

dataset_medhallu_test:
Dataset({
    features: ['label', 'category', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1000
})

dataset_medhalt_test:
Dataset({
    features: ['lab

# **7. BERT model**

In [9]:
bert_classification_model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
for name, param in bert_classification_model.named_parameters():
    print(name)

bert.embeddings.word_embeddings.weight
bert.embeddings.position_embeddings.weight
bert.embeddings.token_type_embeddings.weight
bert.embeddings.LayerNorm.weight
bert.embeddings.LayerNorm.bias
bert.encoder.layer.0.attention.self.query.weight
bert.encoder.layer.0.attention.self.query.bias
bert.encoder.layer.0.attention.self.key.weight
bert.encoder.layer.0.attention.self.key.bias
bert.encoder.layer.0.attention.self.value.weight
bert.encoder.layer.0.attention.self.value.bias
bert.encoder.layer.0.attention.output.dense.weight
bert.encoder.layer.0.attention.output.dense.bias
bert.encoder.layer.0.attention.output.LayerNorm.weight
bert.encoder.layer.0.attention.output.LayerNorm.bias
bert.encoder.layer.0.intermediate.dense.weight
bert.encoder.layer.0.intermediate.dense.bias
bert.encoder.layer.0.output.dense.weight
bert.encoder.layer.0.output.dense.bias
bert.encoder.layer.0.output.LayerNorm.weight
bert.encoder.layer.0.output.LayerNorm.bias
bert.encoder.layer.1.attention.self.query.weight
bert.enc

In [11]:
summary(bert_classification_model)

Layer (type:depth-idx)                                  Param #
BertForSequenceClassification                           --
├─BertModel: 1-1                                        --
│    └─BertEmbeddings: 2-1                              --
│    │    └─Embedding: 3-1                              23,440,896
│    │    └─Embedding: 3-2                              393,216
│    │    └─Embedding: 3-3                              1,536
│    │    └─LayerNorm: 3-4                              1,536
│    │    └─Dropout: 3-5                                --
│    └─BertEncoder: 2-2                                 --
│    │    └─ModuleList: 3-6                             85,054,464
│    └─BertPooler: 2-3                                  --
│    │    └─Linear: 3-7                                 590,592
│    │    └─Tanh: 3-8                                   --
├─Dropout: 1-2                                          --
├─Linear: 1-3                                           1,538
Total params: 10

# **8. Training arguments**

In [12]:
training_args = TrainingArguments(
    output_dir="bert_medhallu",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=10,
    report_to='none'
)

# **9. Metrics**

In [13]:
def compute_metrics(p):
    predictions, labels = p
    preds = np.argmax(predictions, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary'
    )

    acc = (preds == labels).mean()

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

# **10. Trainer on Medhallu datasets**

In [14]:
trainer = Trainer(
    model=bert_classification_model,
    args=training_args,
    train_dataset=dataset_medhallu_train,
    eval_dataset=dataset_medhallu_val,
    compute_metrics=compute_metrics
)

# **11. Train on Medhallu datasets**

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.003812,0.467435,0.916000,0.921201,0.867491,0.982000
2,0.068265,0.339191,0.938000,0.940385,0.905556,0.978000
3,0.132428,0.348105,0.931000,0.933590,0.899814,0.970000
4,0.000994,0.286908,0.951000,0.950254,0.964948,0.936000
5,0.000367,0.335528,0.953000,0.953141,0.950298,0.956000


TrainOutput(global_step=11250, training_loss=0.057284526972213964, metrics={'train_runtime': 4449.6921, 'train_samples_per_second': 20.226, 'train_steps_per_second': 2.528, 'total_flos': 1.184631215652864e+16, 'train_loss': 0.057284526972213964, 'epoch': 5.0})


# **12. First Evaluate on Medhallu dataset (Same datasource as training)**

In [17]:
results = trainer.evaluate(dataset_medhallu_test)
print("\nOverall performance:")
print(results)


Overall performance:
{'eval_loss': 0.37263596057891846, 'eval_accuracy': 0.949, 'eval_f1': 0.9488465396188566, 'eval_precision': 0.9517102615694165, 'eval_recall': 0.946, 'eval_runtime': 15.1001, 'eval_samples_per_second': 66.225, 'eval_steps_per_second': 8.278, 'epoch': 5.0}


# **13. Per-Category Evaluation on Medhallu dataset**

In [18]:
pred_output = trainer.predict(dataset_medhallu_test)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

category_data = defaultdict(lambda: {"y_true": [], "y_pred": []})
dataset_medhallu_test.reset_format()

for true, pred, cat in zip(y_true, y_pred, test_medhallu_categories):
    category_data[cat]["y_true"].append(true)
    category_data[cat]["y_pred"].append(pred)

print("\nPer-Category Performance:")

for cat, data in category_data.items():
    precision, recall, f1, _ = precision_recall_fscore_support(
        data["y_true"],
        data["y_pred"],
        average="binary"
    )

    print(f"\nCategory: {cat}")
    print(f"  Samples:   {len(data['y_true'])}")
    print(f"  Precision: {precision:.3f}")
    print(f"  Recall:    {recall:.3f}")
    print(f"  F1 Score:  {f1:.3f}")


Per-Category Performance:

Category: Misinterpretation of #Question#
  Samples:   742
  Precision: 0.949
  Recall:    0.949
  F1 Score:  0.949

Category: Mechanism and Pathway Misattribution
  Samples:   34
  Precision: 1.000
  Recall:    1.000
  F1 Score:  1.000

Category: Incomplete Information
  Samples:   222
  Precision: 0.963
  Recall:    0.928
  F1 Score:  0.945

Category: Methodological and Evidence Fabrication
  Samples:   2
  Precision: 0.500
  Recall:    1.000
  F1 Score:  0.667


# **14. Evaluate on Med-HALT dataset (with training on Medhallu dataset)**

In [19]:
results_medhalt = trainer.evaluate(dataset_medhalt_test)

print("\nMed-HALT Performance (OOD):")
print(results_medhalt)


Med-HALT Performance (OOD):
{'eval_loss': 3.646761894226074, 'eval_accuracy': 0.4317671430809672, 'eval_f1': 0.3272393750772909, 'eval_precision': 0.8019091873326936, 'eval_recall': 0.20556210106555148, 'eval_runtime': 1779.0504, 'eval_samples_per_second': 64.577, 'eval_steps_per_second': 8.072, 'epoch': 5.0}


# **15. Per-task analysis (Med-HALT)**

In [20]:
from collections import defaultdict

pred_output = trainer.predict(dataset_medhalt_test)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

task_data = defaultdict(lambda: {"y_true": [], "y_pred": []})

dataset_medhalt_test.reset_format()

for t, p, c in zip(y_true, y_pred, test_medhalt_categories):
    task_data[c]["y_true"].append(t)
    task_data[c]["y_pred"].append(p)

print("\nMed-HALT Per-Task Performance:")

for task, data in task_data.items():

    precision, recall, f1, _ = precision_recall_fscore_support(
        data["y_true"],
        data["y_pred"],
        average="binary"
    )

    print(f"\nTask: {task}")
    print(f"  Samples:   {len(data['y_true'])}")
    print(f"  Precision: {precision:.3f}")
    print(f"  Recall:    {recall:.3f}")
    print(f"  F1 Score:  {f1:.3f}")


Med-HALT Per-Task Performance:

Task: FCT
  Samples:   37564
  Precision: 0.501
  Recall:    0.210
  F1 Score:  0.296

Task: NOTA
  Samples:   75464
  Precision: 1.000
  Recall:    0.211
  F1 Score:  0.348

Task: FAKE
  Samples:   1858
  Precision: 0.000
  Recall:    0.000
  F1 Score:  0.000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
